# User Notebook

This notebook demonstrates the three CropGet retrieval modes:

1. Coordinate-based retrieval
2. AEV embedding retrieval
3. Full inference retrieval


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from retrieve import (
    retrieve_with_coords,
    retrieve_with_aev,
    retrieve_inference
)

MODEL_DIR = './models'
CROP = 'maize'


## 1. Coordinate based retrieval

Retrieve similar fields based on location and crop season.

- Works ideally for input coordinates are ALREADY in the reference database

In [ ]:
coordinates = [[48.58, 7.75]]

season_start = 90
season_end = 250

coord_result = retrieve_with_coords(
    coordinates=coordinates,
    season_start=season_start,
    season_end=season_end,
    crop_name=CROP,
    model_dir=MODEL_DIR,
    budget=10
)

coord_result.keys()


In [ ]:
print('Spectra shape:', coord_result['spectra'].shape)
print('Locations:')
print(coord_result['locations'])

print('Distances km:')
print(coord_result['distance_km'])


## 2. AEV retrieval

Retrieve similar fields using learned AEV embeddings.

- Works ideally for ANY coordinates
- Retrieval distance can be used to reject extreme Out Of Distribution retrievals

In [ ]:
coordinates = np.array([[48.58, 7.75]])

# Replace with user AEV input
X = np.load('example_aev.npy')

aev_result = retrieve_with_aev(
    coordinates=coordinates,
    X=X,
    crop_name=CROP,
    model_dir=MODEL_DIR,
    budget=10
)

aev_result.keys()


In [ ]:
print('AEV retrieved spectra:')
print(aev_result['spectra'].shape)

print('Similarity:')
print(aev_result['similarity'])


## 3. CropGet inference

Generate Crop Response Conditioned Climate zones

- Input a raster of AEV features and Crop Name
- Returns a raster, where similar response pixels for a similar would be closeby

In [ ]:
raster_coords = np.array([
    [48.58, 7.75]
])

# Replace with inference features
raster_X = np.load('example_inference_features.npy')

prediction = retrieve_inference(
    raster_coords=raster_coords,
    raster_X=raster_X,
    crop_name=CROP,
    model_dir=MODEL_DIR,
    budget=20
)

prediction.keys()


In [ ]:
expected = prediction['conditioned_spectra']
neighbors = prediction['neighbor_spectra']

print('Expected trajectory:', expected.shape)
print('Retrieved neighbours:', neighbors.shape)


In [ ]:
plt.figure(figsize=(10,4))

plt.plot(
    expected[0,:,3],
    label='CropGet expected trajectory'
)

plt.xlabel('Time')
plt.ylabel('Reflectance')
plt.title('CropGet spectral prediction')
plt.legend()
plt.grid()
plt.show()
